In [ ]:
__file__ = "parser.ipynb"

In [ ]:
import ply.lex as lex

class Lexer:

    reserved = {
        # Sections
        "States": "STATES",
        "Inputs": "INPUTS",
        "Outputs": "OUTPUTS",
        "Initialize": "INITIALIZE",
        "Dynamics": "DYNAMICS",
        "Jacobian": "JACOBIAN",
        "CalcOutputs": "CALC_OUTPUTS",
        # Reserved keywords
        "dt": "DT",
        # C math
        "pow": "POW"
    }
    
    tokens = [
       'COMMENT',
       'IDENTIFIER',
       'LBRACE',
       'RBRACE',
       'LPAREN',
       'RPAREN',
       'LBRACK',
       'RBRACK',
       'SEMICOLON',
        'FLOAT',
        'INTEGER',
        'END',
        'COMMA',
        'PLUS',
        'MINUS',
        'MULTIPLY',
        'DIVIDE',
        'EQUALS',
        'LT',
        'LTE',
        'GT',
        'GTE',
        'QUESTION',
        'COLON',
        'CARET',
        'E',
        ] + list(reserved.values())
    
    t_LBRACE  = r'\{'
    t_RBRACE  = r'\}'
    t_LPAREN  = r'\('
    t_RPAREN  = r'\)'
    t_LBRACK  = r'\['
    t_RBRACK  = r'\]'
    t_COMMA = r'\,'
    t_PLUS = r'\+'
    t_MINUS = r'\-'
    t_MULTIPLY = r'\*'
    t_DIVIDE = r'\/'
    t_EQUALS = r'\='
    t_CARET = r'\^'
    t_QUESTION = r'\?'
    t_COLON = r'\:'
    
    t_SEMICOLON = r'\;'

    t_LT = r'<'
    t_LTE = r'<='
    t_GT = r'>'
    t_GTE = r'>='

    def t_COMMENT(self,t):
        r'\#.*'
        pass
    
    def t_END(self,t):
        r'End\.'
        return t

    def t_IDENTIFIER(self,t):
        r'[a-zA-Z_][a-zA-Z_0-9]*'
        t.type = self.reserved.get(t.value,'IDENTIFIER')    # Check for reserved words
        return t

    def t_E(self,t):
        r'\-?((\d+\.\d*)|(\d*\.\d+))[eE]\-?\d+'
        t.value = float(t.value)
        return t
    
    def t_FLOAT(self,t):
        r'\-?((\d+\.\d*)|(\d*\.\d+))'
        t.value = float(t.value)
        return t

    
    def t_INTEGER(self,t):
        r'\d+'
        t.value = int(t.value)
        return t
    
    def t_newline(self,t):
        r'\n+'
        t.lexer.lineno += len(t.value)
    
    t_ignore  = ' \t'
    
    def t_error(self,t):
        print("Illegal character '%s'" % t.value[0])
        t.lexer.skip(1)

    def __init__(self):
        self.build()
    
    # Build the lexer
    def build(self,**kwargs):
        self.lexer = lex.lex(module=self, **kwargs)

    # Test the output
    def test(self,data):
        self.lexer.input(data)
        while True:
             tok = self.lexer.token()
             if not tok:
                 break
             print(tok)

In [ ]:
import ply.yacc as yacc

class Parser:

    tokens = Lexer.tokens
    start = "model"

    def p_empty(self,p):
        'empty :'
        p[0] = None

    def p_model(self,p):
        'model : sections END'
        p[0] = ("sections",p[1])

    def p_sections(self,p):
        '''sections : sections section
                    | section'''
        p[0] = [*p[1],p[2]] if len(p) == 3 else [p[1]]

    def p_section(self,p):
        '''section : section_1
                   | section_2
                   | statement'''
        p[0] = p[1]

    def p_section_1(self,p):
        'section_1 : section_name_1 EQUALS LBRACE section_content_1 RBRACE SEMICOLON'
        p[0] = ("section",p[1],p[4])

    def p_section_2(self,p):
        'section_2 : section_name_2 LBRACE statements RBRACE'
        p[0] = ("section",p[1],p[3])

    def p_section_name(self,p):
        '''section_name_1 : STATES
                          | INPUTS
                          | OUTPUTS
           section_name_2 : INITIALIZE
                          | DYNAMICS
                          | JACOBIAN
                          | CALC_OUTPUTS'''
        p[0] = p[1]

    def p_section_content_1(self,p):
        '''section_content_1 : delimited_identifiers
                             | delimited_identifiers COMMA
                             | empty'''
        p[0] = p[1]

    def p_delimited_identifiers(self,p):
        '''delimited_identifiers : delimited_identifiers COMMA IDENTIFIER
                                 | IDENTIFIER'''
        p[0] = [*p[1],p[3]] if len(p) == 4 else [p[1]]

    def p_statements(self,p):
        '''statements : statements statement
                      | statement
                      | empty'''
        p[0] = [*p[1],p[2]] if len(p) == 3 else [p[1]]

    def p_statement(self,p):
        'statement : identifier EQUALS expression SEMICOLON'
        p[0] = ("equals",p[1],p[3])

    def p_expression(self,p):
        '''expression : parenthesized_expression
                      | ternary_expression
                      | mathematical_expression
                      | signed_number'''
        p[0] = p[1]

    def p_identifier(self,p):
        '''identifier : DT LPAREN IDENTIFIER RPAREN
                      | IDENTIFIER'''
        p[0] = ("dt", p[3]) if p[1] == "dt" else ("id", p[1])

    def p_parenthesized_expression(self,p):
        'parenthesized_expression : LPAREN expression RPAREN'
        p[0] = p[2]

    def p_ternary_expression(self,p):        
        'ternary_expression : condition QUESTION expression COLON expression'
        p[0] = ("ternary",p[1],p[3],p[5])

    def p_equality_operator(self,p):
        'equality_operator : EQUALS EQUALS'
        p[0] = "=="

    def p_condition_operator(self,p):
        '''condition_operator : equality_operator
                              | LT
                              | LTE
                              | GT
                              | GTE'''
        p[0] = p[1]

    def p_condition(self,p):
        'condition : expression condition_operator expression'
        p[0] = (p[2],p[1],p[3])

    def p_mathematical_expression(self,p):        
        '''mathematical_expression : expression PLUS expression
                                   | expression MINUS expression
                                   | expression MULTIPLY expression
                                   | expression DIVIDE expression
                                   | expression CARET expression'''
        p[0] = (p[2],p[1],p[3])

    def p_mathematical_expression_2(self,p):        
        'mathematical_expression : POW LPAREN expression COMMA expression RPAREN'
        p[0] = (p[1],p[3],p[5])
    
    def p_unsigned_number(self,p):
        '''unsigned_number : INTEGER
                           | FLOAT
                           | E
                           | identifier'''
        p[0] = p[1]

    def p_signed_number(self,p):
        '''signed_number : MINUS unsigned_number
                         | unsigned_number'''
        p[0] = ("minus",p[2]) if p[1] == "-" else ("plus",p[1])
    
    # Error rule for syntax errors
    def p_error(self,p):
        print(f"Syntax error in input! {p}")

    def __init__(self, start=None):
        if start is not None:
            self.start = start
        self.lexer = Lexer()
        self.build()
    
    # Build the parser
    def build(self, **kwargs):
        self.parser = yacc.yacc(module=self, **kwargs)

    # Parse
    def parse(self, data, **kwargs):
        kwargs.setdefault("lexer",self.lexer.lexer)
        return self.parser.parse(data,**kwargs)

In [ ]:
lexer = Lexer()
with open("pred_prey.model", "r") as file:
    lexer.test(file.read())

In [ ]:
def pretty_print_tree(tree, indent=0):
    """
    Recursively prints a tree of tuples with indentation.

    :param tree: The tree structure represented as nested tuples.
    :param indent: The current indentation level.
    """
    if isinstance(tree, (tuple, list)):
        for node in tree:
            if isinstance(node, (tuple,list)):
                pretty_print_tree(node, indent + 4)
            else:
                print(' ' * indent + str(node))
    else:
        print(' ' * indent + str(tree))


In [ ]:
parser = Parser()
with open("naph_pbtk_pde.model", "r") as file:
    pretty_print_tree(parser.parse(file.read()))

In [ ]:
parser = Parser(start="statement")
print(parser.parse("JSC00 = DSC * (CSC01 - CSC00) / SCDX;"))

In [ ]:
TODO
- order of operations

- all reserved keywords (ie functions, etc)
- modulo?

- vectors
- (('A'*'B')>0) for AND (('A'+'B')>0) for OR ('A'==0) for NOT


